## Installation

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

# Model initialization

In [6]:
from unsloth import FastLanguageModel
from unsloth import FastLanguageModel
from datasets import Dataset, load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
import json

# We initialize The Model First Becase We Need The tokenizer

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length= 2048,
    load_in_4bit=True
)

==((====))==  Unsloth 2026.3.11: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


# Data preparation

In [7]:
EOS_TOKEN = tokenizer.eos_token

dataset = load_dataset("json",data_files="tuwaiq_dataset_100.json",split="train")

prompt_template = """السؤال: {}
الإجابة: {}"""

def format_propmts(examples):
    instructions = examples["instruction"]
    outputs = examples["output"]
    texts = []
    for inst, out in zip(instructions,outputs):
        text = prompt_template.format(inst,out) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(format_propmts,batched=True)

print(dataset[2]["text"])

السؤال: هل برامج أكاديمية طويق مجانية؟
الإجابة: نعم، معظم برامج أكاديمية طويق تقدم مجانًا بهدف دعم وتطوير المهارات التقنية.<|eot_id|>


# Trainer

## Body modules
["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

## Head module
["lm_head"]


## Tail (Embeddings Only) module
["embed_tokens"]

In [8]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth"
)

trainer = SFTTrainer(model=model,
                     tokenizer=tokenizer,
                     train_dataset=dataset,
                     dataset_text_field = "text",
                     max_seq_length =2048,
                     args = TrainingArguments(per_device_train_batch_size = 2,
        max_steps = 60,
        learning_rate = 2e-4,
        output_dir = "tuwaiq_head_model",
        optim = "adamw_8bit",)
                     )

trainer.train()

Unsloth 2026.3.11 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 249 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 1 x 1) = 2
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Step,Training Loss


wandb: WARNING URL not available in offline run


train/epoch,▁
train/global_step,▁
total_flos,356453655773184.0
train/epoch,0.48
train/global_step,60
train_loss,1.56696
train_runtime,48.0612
train_samples_per_second,2.497
train_steps_per_second,1.248


TrainOutput(global_step=60, training_loss=1.5669623057047526, metrics={'train_runtime': 48.0612, 'train_samples_per_second': 2.497, 'train_steps_per_second': 1.248, 'total_flos': 356453655773184.0, 'train_loss': 1.5669623057047526, 'epoch': 0.48})

# Inference

In [13]:
FastLanguageModel.for_inference(model)

test_prompt = prompt_template.format(
    "انت مساعد لأكادمية طويق أجب بدقة وبلغة عربية فصحى عن هذا السؤال:أمثلة عن برامج الأكاديمية",
    ""
)

inputs = tokenizer([test_prompt], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)

print("\n--- إجابة النموذج ---")
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128)


--- إجابة النموذج ---
<|begin_of_text|>السؤال: انت مساعد لأكادمية طويق أجب بدقة وبلغة عربية فصحى عن هذا السؤال:أمثلة عن برامج الأكاديمية
الإجابة:  تشمل البرامج التي تقدمها الأكاديمية: برامج حوسبة، وبرامج برمجة، وبرامج تقنية، وبرامج إدارية، وبرامج تدريبية، وبرامج مسابقات، وبرامج تطوير مهارات.<|eot_id|>


# Saving the Model

In [14]:


model.save_pretrained("tuwaiq_head_model")
tokenizer.save_pretrained("tuwaiq_head_model")

print("✅ تم حفظ النموذج بنجاح!")

✅ تم حفظ النموذج بنجاح!


In [15]:
import shutil
shutil.make_archive("tuwaiq_head_model", 'zip', "tuwaiq_head_model")

'/content/tuwaiq_head_model.zip'

In [16]:
# هذا الكود بيدمج المودل الأساسي مع تدريبك ويضغطهم في ملف واحد خفيف
model.save_pretrained_gguf("tuwaiq_model_gguf", tokenizer, quantization_method = "q4_k_m")

Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/896 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [07:17<00:00, 109.34s/it]


Unsloth: Merge process complete. Saved to `/content/tuwaiq_model_gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['tuwaiq_model_gguf_gguf/Llama-3.1-8B-Instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['tuwaiq_model_gguf_gguf/Llama-3.1-8B-Instruct.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model tuwaiq_model_gguf_gguf/Llama-3.1-8B-Instruct.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to tuwaiq_model_gguf_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f tuwaiq_model_gguf_gguf/Modelfile


{'save_directory': 'tuwaiq_model_gguf',
 'gguf_directory': 'tuwaiq_model_gguf_gguf',
 'gguf_files': ['tuwaiq_model_gguf_gguf/Llama-3.1-8B-Instruct.Q4_K_M.gguf'],
 'modelfile_location': 'tuwaiq_model_gguf_gguf/Modelfile',
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': False}